# 🗺️ Análise Geoespacial — Distribuição de Clientes no Brasil

Vamos visualizar onde estão concentrados os clientes do Olist usando mapas interativos com **Folium**.


In [1]:
import sys
sys.path.append('..')

from pathlib import Path
Path('../data/processed').mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

from src.load_data import load_raw, build_master, load_geo

dfs = load_raw()
df  = build_master(dfs)
geo = load_geo(dfs)

print(f'✅ {df.shape[0]:,} pedidos | {geo.shape[0]:,} regiões geográficas')

✅ 110,189 pedidos | 19,015 regiões geográficas


## 1. Mapa de Calor — Densidade de Clientes

In [2]:
df_geo = df.merge(
    geo,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='inner'
).dropna(subset=['geolocation_lat','geolocation_lng'])

df_geo = df_geo[
    (df_geo['geolocation_lat'].between(-35, 5)) &
    (df_geo['geolocation_lng'].between(-75, -30))
]

coords = df_geo[['geolocation_lat','geolocation_lng']].sample(20000, random_state=42).values.tolist()

m = folium.Map(location=[-15.8, -47.9], zoom_start=4, tiles='CartoDB positron')
HeatMap(coords, radius=6, blur=8, min_opacity=0.3).add_to(m)

m.save('../data/processed/mapa_clientes_heatmap.html')
print('✅ Mapa salvo: data/processed/mapa_clientes_heatmap.html')
m

✅ Mapa salvo: data/processed/mapa_clientes_heatmap.html


## 2. Mapa de Receita por Estado (Choropleth)

In [3]:
import urllib.request
import json

GEO_URL = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
try:
    with urllib.request.urlopen(GEO_URL) as r:
        brazil_geo = json.loads(r.read())
    geo_available = True
    print('✅ GeoJSON dos estados carregado')
except Exception:
    geo_available = False
    print('⚠️  Sem acesso à internet — pule esta célula')

if geo_available:
    estado = pd.read_csv('../data/processed/olist_estados.csv')

    m2 = folium.Map(location=[-15.8, -47.9], zoom_start=4, tiles='CartoDB positron')

    folium.Choropleth(
        geo_data=brazil_geo,
        data=estado,
        columns=['customer_state','receita'],
        key_on='feature.properties.sigla',
        fill_color='YlOrRd',
        fill_opacity=0.75,
        line_opacity=0.4,
        legend_name='Receita Total (R$)',
        nan_fill_color='lightgray',
    ).add_to(m2)

    m2.save('../data/processed/mapa_receita_estados.html')
    print('✅ Mapa salvo: data/processed/mapa_receita_estados.html')
    m2

✅ GeoJSON dos estados carregado
✅ Mapa salvo: data/processed/mapa_receita_estados.html


## 3. Top 10 Cidades por Volume de Pedidos

In [4]:
top_cities = (
    df_geo.groupby('customer_city')
    .agg(
        pedidos=('order_id','count'),
        receita=('payment_value','sum'),
        lat=('geolocation_lat','mean'),
        lng=('geolocation_lng','mean'),
    )
    .reset_index()
    .sort_values('pedidos', ascending=False)
    .head(10)
)

m3 = folium.Map(location=[-15.8, -47.9], zoom_start=4, tiles='CartoDB positron')

for _, row in top_cities.iterrows():
    radius = int(np.sqrt(row['pedidos']) * 0.8)
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=max(radius, 5),
        color='#1e40af',
        fill=True,
        fill_color='#3b82f6',
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>{row['customer_city'].title()}</b><br>"
            f"Pedidos: {int(row['pedidos']):,}<br>"
            f"Receita: R$ {row['receita']:,.0f}",
            max_width=200
        ),
        tooltip=row['customer_city'].title(),
    ).add_to(m3)

m3.save('../data/processed/mapa_top_cidades.html')
print('✅ Mapa salvo: data/processed/mapa_top_cidades.html')
print('\nTop 10 cidades:')
print(top_cities[['customer_city','pedidos','receita']].to_string(index=False))
m3

✅ Mapa salvo: data/processed/mapa_top_cidades.html

Top 10 cidades:
        customer_city  pedidos    receita
            sao paulo    17397 2755424.93
       rio de janeiro     7592 1520550.72
       belo horizonte     3087  489662.49
             brasilia     2162  386836.05
             curitiba     1727  322336.65
             campinas     1626  262276.44
         porto alegre     1572  295622.05
             salvador     1351  277021.57
            guarulhos     1293  199200.00
sao bernardo do campo     1041  147467.25
